> **Note:** This environment variable is required for fully deterministic CuBLAS ops on CUDA >= 10.2 when `reproducible=True` is set below. Without it, PyTorch raises a `RuntimeError` instead of training deterministically. It must be set before `torch` is imported. See the [README FAQ](../../README.md#reproducibility-and-cublas_workspace_config) for details.

In [1]:
%env CUBLAS_WORKSPACE_CONFIG=:16:8

env: CUBLAS_WORKSPACE_CONFIG=:16:8


# How to Access Pipeline Results - Deep Dive

Each step in the pipeline writes its results in the `Result` object of the pipeline instance (Vanillix, Varix, etc.).  
In this tutorial, we explore how to access and interpret the results.  

The attributes of the `Result` object are mostly instances of a `TrainingDynamics` class.  
This class provides a standardized interface for accessing results from different splits and epochs.  

**IMPORTANT**  
> Epoch-specific `TrainingDynamics`—such as losses or intermediate latent spaces—are not stored every epoch by default.  
> You need to set the `checkpoint_interval` parameter in the config according to your needs.

## What You Will Learn

We go in depth into:

- The `TrainingDynamics` API  
  - latent spaces  
  - losses  
  - reconstructions  
  - sample_ids  

- Nested `TrainingDynamics` like `sub_losses`  

- Non-`TrainingDynamics` result attributes, such as:  
  - datasets  
  - new_datasets  
  - model  
  - adata_latent  
  - final_reconstruction  
  - embedding_evaluation  

- Special methods to obtain pandas DataFrames  
  - get latent space as a DataFrame with `sample_ids`  
  - get reconstruction as a DataFrame with `sample_ids`  

## 1) Filling the Result Object

Before we can investigate the result object, we first need to create results.  
Therefore, we run two pipelines: `XModalix` and `Varix`.

### 1.1 The Datasets

For the `Varix` example, we use a mock single-cell dataset as a `MuData` object inside our custom `DataPackage` class.  
For our `XModalix` example, we use the same dataset as in the [`XModalix.ipynb`](../PipelineTutorials/XModalix.ipynb) tutorial.  

As a showcase for data modality translation with `XModalix`, we use cancer gene expression from TCGA in combination with handwritten digits from the [MNIST dataset](https://keras.io/api/datasets/mnist/).  
Our goal is to translate the gene expression signature of five selected cancer subtypes to images of digits, where each cancer subtype class is assigned a digit between 0-4.  

In practice, these images could be histopathological images or any other data modality.  
Before showing data preparation and `XModalix` training, here is some background on the basic idea of a cross-modal VAE as proposed by [Yang & Uhler](https://arxiv.org/abs/1902.03515).

### ❗❗ Requirements: Getting Tutorial Data ❗❗

The data for this tutorial is hosted on Hugging Face Hub ([autoencodix/tcga](https://huggingface.co/datasets/autoencodix/tcga)) and is downloaded automatically in the cell below on first run.



In [2]:
import os

p = os.getcwd()
d = "autoencodix_package"
if d not in p:
    raise FileNotFoundError(f"'{d}' not found in path: {p}")
os.chdir(os.sep.join(p.split(os.sep)[: p.split(os.sep).index(d) + 1]))
print(f"Changed to: {os.getcwd()}")


Changed to: /home/janew/ACX/autoencodix_package


In [3]:
# %%capture
from autoencodix.utils.example_data import EXAMPLE_MULTI_SC
from autoencodix.configs.varix_config import VarixConfig
from autoencodix.configs.default_config import DataCase, DataInfo, DataConfig
from autoencodix.configs.xmodalix_config import XModalixConfig
import autoencodix as acx

varix_config = VarixConfig(
    learning_rate=0.001,
    epochs=33,
    checkpoint_interval=1,
    default_vae_loss="kl",  # kl or mmd possible
    data_case=DataCase.MULTI_SINGLE_CELL,
)
varix = acx.Varix(data=EXAMPLE_MULTI_SC, config=varix_config)
result = varix.run()

# XModalix
# ---------------------------------------------------------------------
# Data is hosted on Hugging Face Hub and downloaded (and locally cached)
# automatically, then placed under the paths used below
# ---------------------------------------------------------------------
import shutil
import zipfile
import pandas as pd
from huggingface_hub import hf_hub_download

HF_REPO_ID = "autoencodix/tcga"
xmod_data_root = "data/XModalix-Tut-data"
os.makedirs(xmod_data_root, exist_ok=True)

rna_path = hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename="rna.parquet")
shutil.copyfile(rna_path, os.path.join(xmod_data_root, "combined_rnaseq_formatted.parquet"))

clin_path = hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename="clinical.parquet")
mapping_path = hf_hub_download(
    repo_id=HF_REPO_ID, repo_type="dataset", filename="xmodalix/tcga_image_mappings.txt"
)
# The base clinical file has no image column; the XModalix image mapping is a separate,
# per-architecture extension that we join in here (left join: not every sample has an image).
clin_df = pd.read_parquet(clin_path)
mapping_df = pd.read_csv(mapping_path, sep="\t", index_col="sample_ids")
clin_df["img_paths"] = mapping_df["img_paths"]
clin_df.to_parquet(os.path.join(xmod_data_root, "combined_clin_formatted.parquet"))

images_zip = hf_hub_download(
    repo_id=HF_REPO_ID, repo_type="dataset", filename="xmodalix/tcga_fake_images.zip"
)
images_extract_dir = os.path.join(xmod_data_root, "images")
if not os.path.isdir(os.path.join(images_extract_dir, "tcga_fake")):
    with zipfile.ZipFile(images_zip) as zf:
        zf.extractall(images_extract_dir)

clin_file = os.path.join("data/XModalix-Tut-data/combined_clin_formatted.parquet")
rna_file = os.path.join("data/XModalix-Tut-data/combined_rnaseq_formatted.parquet")
img_root = os.path.join("data/XModalix-Tut-data/images/tcga_fake")

xmodalix_config = XModalixConfig(
    checkpoint_interval=5,
    class_param="CANCER_TYPE_ACRONYM",
    epochs=10,
    data_case=DataCase.IMG_TO_BULK,
    data_config=DataConfig(
        data_info={
            "img": DataInfo(
                file_path=img_root,
                data_type="IMG",
                scaling="MINMAX",
                translate_direction="to",
            ),
            "rna": DataInfo(
                file_path=rna_file,
                data_type="NUMERIC",
                scaling="MINMAX",
                translate_direction="from",
            ),
            "anno": DataInfo(file_path=clin_file, data_type="ANNOTATION", sep="\t"),
        },
    ),
)

xmodalix = acx.XModalix(config=xmodalix_config)
xmodalix_result = xmodalix.run()

in handle_direct_user_data with data: <class 'autoencodix.data.datapackage.DataPackage'>
mudata: View of MuData object with n_obs × n_vars = 1000 × 700
  2 modalities
    rna:	1000 x 500
      obs:	'cell_type', 'batch', 'donor', 'cell_cycle'
    protein:	1000 x 200
      obs:	'cell_type', 'batch', 'donor', 'cell_cycle'
Processing 1 MuData objects: ['multi_sc']
Processing train modality: multi_sc


Processing valid split
Processing valid modality: multi_sc
Processing test split
Processing test modality: multi_sc


You are using a CUDA device ('NVIDIA GeForce RTX 5060 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


/home/janew/ACX/autoencodix_package/src/autoencodix/trainers/_general_trainer.py:295: UserWarning: Gradient clipping was first applied in epoch 0. Total norm of gradients (adjusted for number of features): 12.0150 exceeded max norm of 5.
  warnings.warn(


Epoch 1 - Train Loss: 789.5454
Sub-losses: recon_loss: 789.5454, var_loss: 0.0000, anneal_factor: 0.0000, effective_beta_factor: 0.0000
Epoch 1 - Valid Loss: 861.7886
Sub-losses: recon_loss: 861.7884, var_loss: 0.0003, anneal_factor: 0.0000, effective_beta_factor: 0.0000


Epoch 2 - Train Loss: 714.1906
Sub-losses: recon_loss: 714.1905, var_loss: 0.0001, anneal_factor: 0.0001, effective_beta_factor: 0.0000
Epoch 2 - Valid Loss: 726.2190
Sub-losses: recon_loss: 726.2188, var_loss: 0.0002, anneal_factor: 0.0001, effective_beta_factor: 0.0000


... (41 outputs left out)

--- Epoch 10/10 ---


Epoch 10/10 - Train Loss: 240.8600
Sub-losses - adver_loss: 19.2354, aggregated_sub_losses: 182.4831, paired_loss: 7.0897, class_loss: 32.0518, multi_bulk.rna.recon_loss: 97.8461, multi_bulk.rna.var_loss: 6.0918, multi_bulk.rna.loss: 103.9379, img.img.recon_loss: 72.7175, img.img.var_loss: 5.8277, img.img.loss: 78.5452, clf_loss: 1.0012


Epoch 10/10 - Valid Loss: 229.5862
Sub-losses - adver_loss: 18.2204, aggregated_sub_losses: 172.5583, paired_loss: 7.1107, class_loss: 31.6967, multi_bulk.rna.recon_loss: 86.9691, multi_bulk.rna.var_loss: 5.6081, multi_bulk.rna.loss: 92.5772, img.img.recon_loss: 74.3049, img.img.var_loss: 5.6762, img.img.loss: 79.9811, clf_loss: 0.9996
Storing checkpoint for epoch 9...


Prediction complete.
Processing latent space results into a single AnnData object...
Identified source modality for latent space: 'multi_bulk.rna'
  - Added 17448 source feature IDs to .uns
Finished processing latent results.


## 2) TrainingDynamics Interface Deep Dive
Before accessing the actual results, we provide a theory section on our interface:  
<br><br>
The `TrainingDynamics` object has the following form:  
`<epoch><split><data>`  

So, if you want to access the train loss for the 5th epoch, you would use:  
```python
result.loss.get(epoch=5, split="train")
````

##### The `.get()` Method Explained
Let's say, we're interessted in thre reconstructions of our autoencoder.  
The `reconstructions.get()` method provides flexible access to reconstruction data stored during training. It can retrieve data for specific epochs, specific splits, or any combination of these parameters.

##### Parameters

* **`epoch`** (Optional[int]):

  * Positive integer (e.g., `2`): Get reconstructions from that specific epoch
  * Negative integer (e.g., `-1`): Get the latest epoch (-1), second-to-last (-2), etc.
  * `None`: Return data for all epochs

* **`split`** (Optional[str]):

  * Valid values: `"train"`, `"valid"`, `"test"`
  * `None`: Return data for all splits

##### Return Value Behavior

The method returns different types depending on the parameters:

1. **Both `epoch` and `split` specified**:

   * Returns a NumPy array for that specific epoch and split
   * Example: `get(epoch=2, split="train")` → `array([...])`

2. **Only `epoch` specified**:

   * Returns a dictionary of all splits for that epoch
   * Example: `get(epoch=2)` → `{"train": array([...]), "valid": array([...]), ...}`

3. **Only `split` specified**:

   * Returns a NumPy array containing data for that split across all epochs
   * Example: `get(split="train")` → `array([[...], [...], ...])` (first dimension represents epochs)

4. **Neither specified**:

   * Returns the complete nested dictionary structure
   * Example: `get()` → `{0: {"train": array([...])}, 1: {...}, ...}`

##### Special Handling

* If an invalid split is provided, a `KeyError` is raised
* Negative epoch indices work like Python list indexing (-1 is the last epoch)
* If an epoch doesn't exist, an empty array or dictionary is returned

##### Code Example

```python
# Access train reconstructions for the 5th epoch
train_epoch_5 = result.reconstructions.get(epoch=5, split="train")

# Access all splits for the latest epoch
latest_epoch_all_splits = result.reconstructions.get(epoch=-1)

# Access data for all epochs for the "valid" split
all_epochs_valid = result.reconstructions.get(split="valid")

# Access the full nested dictionary
full_data = result.reconstructions.get()
```


## 3) Working with Actual Results
### 3.1) Varix

Exemplary, we show how to get the `latentspaces` of the `result` attribute and to access different loss types.

In [4]:
all_ls = result.latentspaces.get()
print(f"Keys of all latentspaces: {all_ls.keys()}")


Keys of all latentspaces: dict_keys([-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32])


We see that we have latent spaces for each epoch because we set `checkpoint_interval=1` in our configs in [Step 1](#1-filling-the-result-object).  

For each epoch, we have the latent space for the `train` and `valid` splits. The `-1` epoch is a special key for the `test` split. For the other splits, negative indexing works as in Python lists: `-1` gives the last epoch, `-2` the second-to-last, and so on.  

**Special Case**  
> You cannot get the last epoch for all splits at once. You can either get the last epoch for `train` and `valid`, or only for `test`. See code below.


In [5]:
print(f"Splits in 2. epoch: {result.latentspaces.get(epoch=2).keys()}")

# this will only give you the data for train and valid, since 'test' is a special case
print(f"Splits in epoch =-1: {result.latentspaces.get(epoch=-1).keys()}")
# Get test by adding the 'split' argument.
test_ls = result.latentspaces.get(split="test")
print(test_ls[0][0])
# get a specific epoch and specific split:

print("\n")
print("-"*80)

print(f"latentspace of one sample in train split at epoch 4: {result.latentspaces.get(split='train', epoch=4)[0]}")

Splits in 2. epoch: dict_keys(['train', 'valid'])
Splits in epoch =-1: dict_keys(['train', 'valid'])
[ 3.6223373   0.09076981  0.9311101   0.4686319   0.5648282   4.362979
  7.462656    4.2597566  -3.0950072   5.624643   -0.7086104  -1.9337438
 -0.5916452  -0.30150637  6.15409     7.991743  ]


--------------------------------------------------------------------------------
latentspace of one sample in train split at epoch 4: [-0.7468408   2.9747071   0.17302047  0.04815996 -2.0036192   0.17140199
 -1.1278459   4.9971147   2.4795966  -0.19799425  2.1384008   2.5776906
 -0.8080328  -0.23301628 -0.6225459   2.2121964 ]


### 3.2 XModalix

Accessing the results works slightly differently for `XModalix`, because we have:

1. **Multiple latent spaces** (one for each data modality).  

2. **Translations and reference translations**, not just reconstructions.  
   - These are reconstructions within one data modality on the same split as the translation. For example, if we translate from `rna` to `img`, the reference translation would be the reconstruction from `img` to `img`.  
   - You can access these `translations` via the `reconstruction` attribute of the `result` object. First, apply the usual `TrainingDynamics` API via `get()`, then you get a dictionary for each data modality with translations and reference translations.

3. **Multiple losses**.  
   - These are accessed via the `sub_losses` attribute of the `result` object, which is a dictionary of `TrainingDynamics`. First, select the loss type you're interested in, then work with the usual `TrainingDynamics` API.  

   **Note on naming sub-losses:**  
   - Global losses are simply named after the loss type, e.g., `class_loss`.  
   - Losses per data modality are named with the following convention:  
     ```
     <global_data_modality_key>.<specific_data_modality_name>.<loss_name>
     ```  
     For example: `multi_bulk.rna.class_loss` or `multi_sc.celltype.reconstruction_loss`. See print below.


#### 3.2.1 Access Modality Latent Spaces

As described in our `XModalix Deep Dive` [1], we fit one latent space per data modality.  
You can access this by first selecting the epoch and split you're interested in (standard `TrainingDynamics` API).  
The result will be a `Dict` with the name of each data modality as the key.


In [6]:
print(f" Keys of data modalities for latent space dynamic: {xmodalix_result.latentspaces.get(epoch=-1, split='test').keys()}")


 Keys of data modalities for latent space dynamic: dict_keys(['multi_bulk.rna', 'img.img'])


Now you can access the latent space of the `image` modality with the key `img.img`

In [7]:
xmodalix_result.latentspaces.get(epoch=-1, split="test").get("img.img")

array([[ 0.8092547 ,  4.121351  ,  1.4299108 , ...,  5.3962545 ,
         0.6538164 ,  0.9702057 ],
       [ 2.1221623 ,  5.256636  ,  0.04541756, ...,  5.646598  ,
         0.04192398,  1.9187347 ],
       [ 5.382866  ,  6.639792  , -0.14126651, ...,  6.236802  ,
        -1.1598693 ,  2.861813  ],
       ...,
       [ 1.267178  , -2.6413867 ,  0.61661077, ..., -0.13882025,
        -1.8977132 , 12.516397  ],
       [ 7.283764  , -1.0312403 , -1.7102883 , ...,  3.6146328 ,
        -0.6609394 ,  4.1691995 ],
       [ 1.247888  ,  4.62085   ,  1.3288131 , ...,  4.4530516 ,
        -0.98057437,  0.6013475 ]], shape=(646, 16), dtype=float32)

#### 3.2.2 Access Translation

In [8]:
print("Get reconstruction keys")
# Frist define split and epoch you're interested in
# usually test split (there are no epochs, so by default this is always epoch=-1)
recons = xmodalix_result.reconstructions.get(split="test", epoch=-1)
print(recons.keys())
print("Getting Translation")
trans = recons.get("translation")
print(f"shape of translation: {trans.shape}")

Get reconstruction keys
dict_keys(['multi_bulk.rna', 'img.img', 'translation', 'reference_img.img_to_img.img'])
Getting Translation
shape of translation: (711, 1, 64, 64)


#### 3.2.3 Access Sub-Losses

In [9]:

sub_losses = xmodalix_result.sub_losses
print("Sub Losses:")
print(f"keys: {sub_losses.keys()}")
print("\n")
recon_dyn = sub_losses.get(key="paired_loss")
print("Value of paired loss in epoch 4 for train split")
print(f"{recon_dyn.get(split='train', epoch=4):.2f}")


Sub Losses:
keys: dict_keys(['adver_loss', 'aggregated_sub_losses', 'paired_loss', 'class_loss', 'multi_bulk.rna.recon_loss', 'multi_bulk.rna.var_loss', 'multi_bulk.rna.anneal_factor', 'multi_bulk.rna.effective_beta_factor', 'multi_bulk.rna.loss', 'img.img.recon_loss', 'img.img.var_loss', 'img.img.anneal_factor', 'img.img.effective_beta_factor', 'img.img.loss', 'clf_loss'])


Value of paired loss in epoch 4 for train split
7.75


## 4) Non-TrainingDynamics Result Attributes

There are other (intermediate) results that are not created during training but might still be interesting.  
These results do not follow a uniform interface like `TrainingDynamics`, but are often more straightforward. We go over each attribute quickly.

#### 4.1 Datasets

The `datasets` attribute stores the preprocessed data in a `DatasetContainer`.  
This is basically a dict with `train`, `valid`, and `test` as keys, and each value is a child class of a PyTorch dataset.  
Whenever you need to re-access your preprocessed data, you can do so using the `datasets` attribute, as shown below:


In [10]:
print(result.datasets)
print(result.datasets.train)
print(type(result.datasets.train.data))
result.datasets.train.metadata.head()

DatasetContainer(train=<autoencodix.data._numeric_dataset.NumericDataset object at 0x7c930fb41b80>, valid=<autoencodix.data._numeric_dataset.NumericDataset object at 0x7c930fc8e0f0>, test=<autoencodix.data._numeric_dataset.NumericDataset object at 0x7c933c175cd0>)
<class 'torch.Tensor'>


,cell_type,batch,donor,cell_cycle,n_genes
cell_0,type_0,batch2,donor2,S,346
cell_1,type_0,batch3,donor4,G1,357
cell_19,type_3,batch1,donor1,G1,345
cell_190,type_4,batch1,donor2,S,338
cell_191,type_0,batch3,donor2,S,349


#### 4.2 New Datasets

Whenever you run the `predict` step of the pipeline and pass new, unseen data to it, we preprocess this data (if necessary).  
To avoid overwriting the original `datasets`, we store this in `new_datasets`.  

If you run `predict` again with other data, `new_datasets` will be overridden.  
Otherwise, `new_datasets` works the same way as `datasets`.


First we create new data and then we run predict.

In [11]:
import copy

from autoencodix.utils.example_data import EXAMPLE_MULTI_SC
new_data = copy.copy(EXAMPLE_MULTI_SC)
new_multi_sc = new_data.multi_sc["multi_sc"]
for modname, mod in new_multi_sc.mod.items():
    new_names = mod.obs_names.str.replace('cell', 'new_cell')
    mod.index = new_names

    mod.obs_names = new_names
    print(mod.index)
    new_multi_sc.mod[modname] = mod

new_data.multi_sc["multi_sc"] = new_multi_sc
new_data.multi_sc["multi_sc"].update()

Index(['new_cell_0', 'new_cell_1', 'new_cell_10', 'new_cell_100',
       'new_cell_101', 'new_cell_102', 'new_cell_103', 'new_cell_104',
       'new_cell_105', 'new_cell_106',
       ...
       'new_cell_990', 'new_cell_991', 'new_cell_992', 'new_cell_993',
       'new_cell_994', 'new_cell_995', 'new_cell_996', 'new_cell_997',
       'new_cell_998', 'new_cell_999'],
      dtype='object', length=1000)
Index(['new_cell_0', 'new_cell_1', 'new_cell_10', 'new_cell_100',
       'new_cell_101', 'new_cell_102', 'new_cell_103', 'new_cell_104',
       'new_cell_105', 'new_cell_106',
       ...
       'new_cell_990', 'new_cell_991', 'new_cell_992', 'new_cell_993',
       'new_cell_994', 'new_cell_995', 'new_cell_996', 'new_cell_997',
       'new_cell_998', 'new_cell_999'],
      dtype='object', length=1000)


.venv/lib/python3.12/site-packages/mudata/_core/mudata.py:758: ImplicitModificationWarning: Setting element `.varm['rna']` of view, initializing view as actual.
  attrm[mod] = mapping > 0
.venv/lib/python3.12/site-packages/mudata/_core/mudata.py:758: ImplicitModificationWarning: Setting element `.varm['protein']` of view, initializing view as actual.
  attrm[mod] = mapping > 0


Run the predict step:

In [12]:
%%capture
varix.predict(data=new_data)

Examine `datasets` and `new_datasets`:  
We see that `datasets` still is kept and only new_datasets is updated.

In [13]:
print(f"Sample of original dataset: {result.datasets.train.sample_ids[0]}")

print(f"Sample of new dataset: {result.new_datasets.test.sample_ids[0]}")

Sample of original dataset: cell_0
Sample of new dataset: new_cell_0


#### 4.3 Model Attribute
This is straightforward the trained model as PytTorch Module.

In [14]:
result.model

VarixArchitecture(
  (_encoder): Sequential(
    (0): Linear(in_features=700, out_features=175, bias=True)
    (1): BatchNorm1d(175, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Dropout(p=0.1, inplace=False)
    (3): ReLU()
    (4): Linear(in_features=175, out_features=43, bias=True)
    (5): BatchNorm1d(43, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): Dropout(p=0.1, inplace=False)
    (7): ReLU()
    (8): Linear(in_features=43, out_features=16, bias=True)
    (9): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): Dropout(p=0.1, inplace=False)
    (11): ReLU()
  )
  (_mu): Linear(in_features=16, out_features=16, bias=True)
  (_logvar): Linear(in_features=16, out_features=16, bias=True)
  (_decoder): Sequential(
    (0): Linear(in_features=16, out_features=16, bias=True)
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Dropout(p=0.1, inplace=Fa

#### 4.4 Adata Latent

We save the latent space of the `test` split from the final trained model as an `AnnData` object for the single-cell community.  

This is also useful for non-single-cell cases, because you can still obtain the sample IDs via `.obs`.


In [15]:
print(result.adata_latent)
print(result.adata_latent.obs)

AnnData object with n_obs × n_vars = 1000 × 16
    uns: 'var_names'
Empty DataFrame
Columns: []
Index: [new_cell_0, new_cell_1, new_cell_10, new_cell_100, new_cell_101, new_cell_102, new_cell_103, new_cell_104, new_cell_105, new_cell_106, new_cell_107, new_cell_108, new_cell_109, new_cell_11, new_cell_110, new_cell_111, new_cell_112, new_cell_113, new_cell_114, new_cell_115, new_cell_116, new_cell_117, new_cell_118, new_cell_119, new_cell_12, new_cell_120, new_cell_121, new_cell_122, new_cell_123, new_cell_124, new_cell_125, new_cell_126, new_cell_127, new_cell_128, new_cell_129, new_cell_13, new_cell_130, new_cell_131, new_cell_132, new_cell_133, new_cell_134, new_cell_135, new_cell_136, new_cell_137, new_cell_138, new_cell_139, new_cell_14, new_cell_140, new_cell_141, new_cell_142, new_cell_143, new_cell_144, new_cell_145, new_cell_146, new_cell_147, new_cell_148, new_cell_149, new_cell_15, new_cell_150, new_cell_151, new_cell_152, new_cell_153, new_cell_154, new_cell_155, new_cell_1

#### 4.5 Final Reconstruction
This attribute gives you the exact data structure as you used for input i.e. `MuData` in our case, with the reconstructed values.

In [16]:
result.final_reconstruction

MuData object with n_obs × n_vars = 1000 × 700
  2 modalities
    rna:	1000 x 500
      obs:	'cell_type', 'batch', 'donor', 'cell_cycle', 'n_genes'
    protein:	1000 x 200
      obs:	'cell_type', 'batch', 'donor', 'cell_cycle', 'n_genes'

#### 4.6 Evaluation Embeddings

Before we can access this attribute, we first need to run the `evaluate` step. This will use the latent space and train a downstream machine learning task. In our case, we want to classify the cancer type.  

The results of this evaluate step will be stored in `embedding_evaluation`.


In [17]:
%%capture
xmodalix.evaluate(params=["CANCER_TYPE"])

In [18]:
xmodalix_result.embedding_evaluation

,score_split,CLINIC_PARAM,metric,value,ML_ALG,ML_TYPE,MODALITY,ML_TASK,ML_SUBTASK
0,train,CANCER_TYPE,roc_auc_ovo,0.995432,LogisticRegression(),classification,multi_bulk.rna,Latent,Latent_$_multi_bulk.rna
1,valid,CANCER_TYPE,roc_auc_ovo,0.998581,LogisticRegression(),classification,multi_bulk.rna,Latent,Latent_$_multi_bulk.rna
2,test,CANCER_TYPE,roc_auc_ovo,0.995974,LogisticRegression(),classification,multi_bulk.rna,Latent,Latent_$_multi_bulk.rna
0,train,CANCER_TYPE,roc_auc_ovo,0.993840,LogisticRegression(),classification,img.img,Latent,Latent_$_img.img
1,valid,CANCER_TYPE,roc_auc_ovo,0.986195,LogisticRegression(),classification,img.img,Latent,Latent_$_img.img
2,test,CANCER_TYPE,roc_auc_ovo,0.986864,LogisticRegression(),classification,img.img,Latent,Latent_$_img.img


In [19]:

varix.result.datasets.test.metadata.head()
varix.evaluate(params=["batch"])

Perform ML task with feature df: Latent
Latent
Perform ML task for target parameter: batch
Perform ML task with feature df: Latent
Latent
Perform ML task for target parameter: batch


.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Result Object Public Attributes:
------------------------------
latentspaces: TrainingDynamics object
sample_ids: TrainingDynamics object
reconstructions: TrainingDynamics object
mus: TrainingDynamics object
sigmas: TrainingDynamics object
losses: TrainingDynamics object
sub_losses: LossRegistry(_losses={'recon_loss': TrainingDynamics(), 'var_loss': TrainingDynamics(), 'anneal_factor': TrainingDynamics(), 'effective_beta_factor': TrainingDynamics()})
preprocessed_data: Tensor of shape (0,)
model: VarixArchitecture
model_checkpoints: TrainingDynamics object
class_means: TrainingDynamics object
datasets: DatasetContainer(train=<autoencodix.data._numeric_dataset.NumericDataset object at 0x7c933c70fbf0>, valid=<autoencodix.data._numeric_dataset.NumericDataset object at 0x7c930fc8e0f0>, test=<autoencodix.data._numeric_dataset.NumericDataset object at 0x7c933c7b1400>)
new_datasets: DatasetContainer(train=<autoencodix.data._numeric_dataset.NumericDataset object at 0x7c933c70fbf0>, valid=None,

##  5 Special Methods to Obtain DataFrames
As you've  seen in the [Training Dynamics Section](#2-trainingdynamics-interface-deep-dive), we only get plain values of reconstructions and latetnspaces. Often it is more useful to have sample ids, too. We could obtain the sample ids in the same order via the `sample_ids` TrainingDynamic. To make this more accessible, we added the methods
`get_latent_df` and `get_reconstructions_df`. Here you pass `epoch` and `split` as seen before and you get a pandas DataFrame for the specific split and epoch for the latent space or the reconstruction.

In [20]:
result.get_latent_df(epoch=-1, split="test").head()

,LatDim_0,LatDim_1,LatDim_2,LatDim_3,LatDim_4,LatDim_5,LatDim_6,LatDim_7,LatDim_8,LatDim_9,LatDim_10,LatDim_11,LatDim_12,LatDim_13,LatDim_14,LatDim_15
new_cell_0,1.169099,6.428045,-0.909395,0.374270,0.522860,-0.824939,1.320667,0.916621,3.864682,0.547017,8.015424,4.431547,-0.753605,-1.017359,0.735275,4.925269
new_cell_1,3.025050,6.690741,0.612838,-0.001472,0.647654,-1.513513,1.393777,-1.178969,1.668167,-1.575379,9.334797,3.258978,1.502755,0.931644,-0.203693,5.011363
new_cell_10,-0.952251,0.109878,0.152486,0.666301,-0.460282,0.398765,-0.068070,1.201444,-0.069014,-1.078704,1.034621,1.796616,0.391577,0.868087,-0.355039,0.085928
new_cell_100,-0.601422,-2.526164,-0.972690,0.702619,2.239834,0.029732,0.817120,0.423556,-0.162558,0.135305,0.155343,0.305283,-0.373023,0.563850,-0.910124,-0.620062
new_cell_101,0.452678,4.728966,2.322374,-0.792972,0.057138,-0.416141,0.811696,0.929577,0.130278,-0.204889,7.578441,3.152101,0.249232,0.179789,-2.100044,2.799470


In [21]:
result.get_reconstructions_df(epoch=-1, split="test").head()

,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,protein_190,protein_191,protein_192,protein_193,protein_194,protein_195,protein_196,protein_197,protein_198,protein_199
new_cell_0,1.182295,-0.796981,0.030274,1.458777,-0.325000,0.197361,-0.000707,-0.019382,-0.742352,-0.733279,...,-1.050608,1.612186,0.162170,-0.430544,-0.057345,-1.153803,-0.433501,0.075737,-1.191396,1.251562
new_cell_1,1.172235,-0.790350,0.034178,1.442763,-0.313463,0.197347,-0.000446,-0.024263,-0.737805,-0.726801,...,-1.042170,1.587032,0.162941,-0.421808,-0.042026,-1.147030,-0.442250,0.061307,-1.191653,1.268421
new_cell_10,-0.613804,0.630371,-0.016076,-0.494771,-0.308604,-0.622271,-0.010671,-0.022798,0.682072,0.599002,...,0.906552,-0.558953,-0.075986,-0.452418,-0.095015,0.129426,-0.444400,-0.025997,0.908695,-0.706898
new_cell_100,-0.604726,0.621461,-0.011855,-0.480109,-0.313499,-0.623201,-0.006275,-0.018529,0.677175,0.585029,...,0.890978,-0.543339,-0.069825,-0.447965,-0.099827,0.109967,-0.449088,-0.037517,0.899632,-0.703618
new_cell_101,1.127195,-0.766829,0.018688,1.408473,-0.293719,0.224175,0.025476,-0.023315,-0.749017,-0.687181,...,-1.021974,1.513067,0.180992,-0.366846,-0.048300,-1.050626,-0.366179,0.044433,-1.156877,1.206789
